# 05 -- Megadrought Analysis: Climate Drivers, Recession, Aridity, and Long-Term Trends

Merged from `05_megadrought_analysis` and `06_megadrought_extensions`. Runs on one basin set: the final basins from `02_final_basin_set.ipynb`, with results from `03_glacier_streamflow_fraction.ipynb` (run both first; run `04_regression_analysis.ipynb` for the regression panels).

The drought amplification and decomposition (old 05 Sections IV and V), the lag analysis, lag-corrected fractions and peak water (old 06) are now in `06_key_findings_summary.ipynb`.

Sections
- **Climate drivers (basin-average CAMELS-CL, OGGM snowfall):** I. Load climate, II. Empirical validation of drought period definitions, III. Snow drought diagnosis, IV. Climate driver analysis
- **Becca:** recession coefficients, routing, spatial map, basin aridity, long-term aridity trend
- **Noah:** OGGM melt trend, mass-balance "bank account" test, melt/precip trend table, Ayala replication check, MB decomposition
- **Sanity checks:** mu_star comparison (Ale), seasonal definitions
- **Summary** of findings (old text, to be rewritten)

**Written findings (Summary section) come from the old 143-basin run. Rewrite them from the new output before citing.**

## Cell 0: Imports, paths, and data loading

In [ ]:
import os, pickle, calendar, warnings, json as jsonlib
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.colors import Normalize, to_hex
from matplotlib.cm import ScalarMappable
from scipy import stats
from scipy.stats import linregress
from scipy.optimize import curve_fit
warnings.filterwarnings('ignore')

# One basin set: the final basins from 02_final_basin_set.ipynb.
# Results come from 03_glacier_streamflow_fraction.ipynb.
BASIN_SET = 'final_basin_set'

BASE_PATH    = '/Users/milliespencer/v2_CR2_OGGM_project_11_2026'
OUTPUT_ROOT  = f'{BASE_PATH}/Output'
CAMELS_FULL  = f'{BASE_PATH}/Full_CAMELS_all_basins/CAMELS_CL_v202201'
RGI_PATH     = f'{BASE_PATH}/files_chile_OGGM_climate_comparison/RGI_BNA_Clusters.csv'
BOUNDS_PATH  = f'{CAMELS_FULL}/camels_cl_boundaries/camels_cl_boundaries.shp'

FINAL_DIR    = f'{OUTPUT_ROOT}/final_basin_set'          # from 02
OUT_DIR      = f'{OUTPUT_ROOT}/results_final_basins'     # from 03
REG_DIR      = f'{OUTPUT_ROOT}/regression/{BASIN_SET}'   # from 04
DROUGHT_DIR  = f'{OUTPUT_ROOT}/megadrought_analysis/{BASIN_SET}'
os.makedirs(DROUGHT_DIR, exist_ok=True)

YEARS         = list(range(2000, 2020))
MONTHS        = list(range(1, 13))
YEAR_OFFSET   = 1
WATER_DENSITY = 1000
CLUSTERS      = ['OT3','DA1','DA2','DA3','WA1','WA2','WA3','WA4','WA5','WA6']
DATASETS      = {'CR2MET': 'TC', 'ERA5': 'ERA5', 'CRU': 'CRU'}
DS_COLORS     = {'CR2MET': '#1f77b4', 'ERA5': '#ff7f0e', 'CRU': '#2ca02c'}
REGION_LABELS = {
    'OT3': 'Altiplano',     'DA1': 'Dry Andes N',
    'DA2': 'Dry Andes C',   'DA3': 'Dry Andes S',
    'WA1': 'Wet Andes N',   'WA2': 'Wet Andes C',
    'WA3': 'N Patagonia',   'WA4': 'S Patagonia',
    'WA5': 'Patagonia',     'WA6': 'Tierra del Fuego',
}
BASELINE     = list(range(2000, 2010))
DROUGHT_DEFS = {
    'Baseline (2000-2009)':  list(range(2000, 2010)),
    'Drought 1 (2010-2019)': list(range(2010, 2020)),
    'Drought 2 (2013-2019)': list(range(2013, 2020)),
    'Drought 3 (2015-2019)': list(range(2015, 2020)),
}
DROUGHT_COLORS = {
    'Baseline (2000-2009)':  '#2ecc71',
    'Drought 1 (2010-2019)': '#f39c12',
    'Drought 2 (2013-2019)': '#e67e22',
    'Drought 3 (2015-2019)': '#c0392b',
}
SUMMER = [12, 1, 2]
WINTER = [6, 7, 8]
CENTRAL_ANDES = ['DA1','DA2','DA3']

def nc_path(dataset_name, cluster):
    tag = DATASETS[dataset_name]
    return (f'{OUTPUT_ROOT}/{dataset_name}/{cluster}/'
            f'run_output_2000_2019_hydro_{tag}_{cluster}.nc')

# ── Load RESULTS and the final basin set ────────────────────────────────────
with open(f'{OUT_DIR}/RESULTS.pkl', 'rb') as f:
    RESULTS = pickle.load(f)
print(f'RESULTS loaded: {len(RESULTS["CR2MET"])} gauges')

assign      = pd.read_csv(f'{FINAL_DIR}/glacier_assignment.csv', dtype={'gauge_id': str})
final_table = pd.read_csv(f'{FINAL_DIR}/final_basin_table.csv', dtype={'gauge_id': str})
rgi_all_df  = pd.read_csv(RGI_PATH)

rgi_ids_by_gauge = assign.groupby('gauge_id')['RGIId'].apply(list).to_dict()
cen_lon_by_gauge = (assign[['RGIId', 'gauge_id']]
                    .merge(rgi_all_df[['RGIId', 'CenLon']], on='RGIId')
                    .groupby('gauge_id')['CenLon'].mean().to_dict())

GAUGE_INFO_FRAC = {}
for gauge_id, res in RESULTS['CR2MET'].items():
    if res is None or gauge_id not in set(final_table['gauge_id']):
        continue
    GAUGE_INFO_FRAC[gauge_id] = {
        'gauge_name':       res['gauge_name'],
        'rgi_ids':          rgi_ids_by_gauge.get(gauge_id, []),
        'n_glaciers':       res['n_glaciers'],
        'glacier_area_km2': res['glacier_area_km2'],
        'basin_area_km2':   res['basin_area_km2'],
        'clusters':         res['clusters'],             # main zone first
        'cen_lat':          res['cen_lat'],
        'cen_lon':          cen_lon_by_gauge.get(gauge_id, np.nan),
        'complete_years':   res['complete_years'],
    }

# Basins dropped by the high-fraction check in notebook 03
_excl = f'{OUT_DIR}/excluded_high_frac_gauges.csv'
if os.path.exists(_excl):
    excluded = pd.read_csv(_excl, dtype={'gauge_id': str})
    for gauge_id in excluded['gauge_id'].tolist():
        GAUGE_INFO_FRAC.pop(gauge_id, None)
    print(f'Dropped by the high-fraction check: {len(excluded)}')

BASIN_SET_LABEL = f'Final basins (n={len(GAUGE_INFO_FRAC)})'
print(BASIN_SET_LABEL)
print(f'Outputs will be saved to: {DROUGHT_DIR}')
print('Cell 0 complete')

## I. Load CAMELS-CL basin-average climate data

Basin-average monthly precipitation, temperature, and PET from
CAMELS-CL (CR2MET). Used to empirically validate drought period
definitions and diagnose drought drivers -- covering all 516 national CAMELS gauges.

In [ ]:
print('Loading CAMELS-CL climate...')

def load_camels_mon(fname, year_range=(2000,2019)):
    df = pd.read_csv(f'{CAMELS_FULL}/{fname}')
    df = df[(df['year'] >= year_range[0]) & (df['year'] <= year_range[1])].copy()
    gauge_cols = [c for c in df.columns if c not in ['date','year','month','day']]
    return df[['year','month'] + gauge_cols]

precip_df = load_camels_mon('precip_cr2met_mm_mon.csv')
tmean_df  = load_camels_mon('tmean_cr2met_C_mon.csv')
pet_df    = load_camels_mon('pet_hargreaves_mm_mon.csv')

print(f'Precip: {precip_df.shape}  Tmean: {tmean_df.shape}  PET: {pet_df.shape}')

In [ ]:
# Bounds and country borders -- needed by the spatial map cell below.
bounds = gpd.read_file(BOUNDS_PATH)
bounds['gauge_id'] = bounds['gauge_id'].astype(str)

try:
    world = gpd.read_file(
        'https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip')
    chile = world[world['NAME'] == 'Chile']
    arg   = world[world['NAME'] == 'Argentina']
    has_borders = True
except Exception:
    has_borders = False
    print('Country borders unavailable')

print(f'Bounds loaded: {len(bounds)} CAMELS basins')

## II. Empirical validation of drought period definitions

Confirms the three drought period definitions correspond to real
precipitation deficits, using basin-average CR2MET annual/seasonal
anomalies vs. the 2000-2009 baseline. Computed once using the
`all_glacierized` gauge set (broadest per-cluster station sample) --
see design note at the top of this notebook.

In [ ]:
NON_GAUGE = ['date', 'year', 'month', 'day']

cluster_gauges = {cl: [gid for gid, info in GAUGE_INFO_FRAC.items() if cl in info['clusters']]
                  for cl in CLUSTERS}

print('Computing annual climate aggregations...')

def season_agg(df, months, agg='sum'):
    gauge_cols = [c for c in df.columns if c not in NON_GAUGE]
    sub = df[df['month'].isin(months)][['year'] + gauge_cols]
    if agg == 'sum':
        return sub.groupby('year')[gauge_cols].sum(numeric_only=True)
    return sub.groupby('year')[gauge_cols].mean(numeric_only=True)

ann_precip = season_agg(precip_df, list(range(1,13)), 'sum')
win_precip = season_agg(precip_df, WINTER, 'sum')
ann_tmean  = season_agg(tmean_df, list(range(1,13)), 'mean')
ann_pet    = season_agg(pet_df, list(range(1,13)), 'sum')

clim_records = []
for cl in CLUSTERS:
    gauges = [g for g in cluster_gauges[cl] if g in ann_precip.columns]
    if not gauges:
        print(f'  {cl}: no gauges found -- skipping')
        continue
    for year in YEARS:
        def get_val(df_wide, yr):
            if yr not in df_wide.index:
                return np.nan
            cols = [g for g in gauges if g in df_wide.columns]
            return df_wide.loc[yr, cols].mean() if cols else np.nan

        clim_records.append({
            'cluster': cl, 'year': year,
            'precip_ann': get_val(ann_precip, year), 'precip_win': get_val(win_precip, year),
            'tmean_ann': get_val(ann_tmean, year), 'pet_ann': get_val(ann_pet, year),
        })

df_clim = pd.DataFrame(clim_records)
print(f'Climate DataFrame built: {df_clim.shape}')

baseline_means = (df_clim[df_clim['year'].isin(BASELINE)].groupby('cluster')[
    ['precip_ann','precip_win','tmean_ann','pet_ann']].mean().rename(columns=lambda c: c + '_base'))

df_clim = df_clim.merge(baseline_means, on='cluster', how='left')
df_clim['precip_ann_anom_pct'] = ((df_clim['precip_ann'] - df_clim['precip_ann_base']) / df_clim['precip_ann_base'] * 100)
df_clim['precip_win_anom_pct'] = ((df_clim['precip_win'] - df_clim['precip_win_base']) / df_clim['precip_win_base'] * 100)
df_clim['tmean_anom']          = (df_clim['tmean_ann'] - df_clim['tmean_ann_base'])
df_clim['pet_anom_pct']        = ((df_clim['pet_ann'] - df_clim['pet_ann_base']) / df_clim['pet_ann_base'] * 100)

print()
print('Annual precipitation anomaly vs 2000-2009 baseline (%) by period:')
print(f'{"Cluster":6s}', end='')
for period_label in list(DROUGHT_DEFS.keys())[1:]:
    print(f'  {period_label:22s}', end='')
print()
print('-' * 80)

for cl in CLUSTERS:
    cl_df = df_clim[df_clim['cluster'] == cl]
    if len(cl_df) == 0:
        continue
    print(f'{cl:6s}', end='')
    for period_label, years in list(DROUGHT_DEFS.items())[1:]:
        anom = cl_df[cl_df['year'].isin(years)]['precip_ann_anom_pct'].mean()
        print(f'  {anom:+.1f}%{" "*18}', end='')
    print()

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True)
axes_flat = axes.flatten()

for ax, cl in zip(axes_flat, CLUSTERS):
    cl_df = df_clim[df_clim['cluster'] == cl].set_index('year')
    if len(cl_df) == 0:
        ax.set_visible(False)
        continue
    colors = ['#c0392b' if v < 0 else '#2980b9' for v in cl_df['precip_ann_anom_pct']]
    ax.bar(cl_df.index, cl_df['precip_ann_anom_pct'], color=colors, alpha=0.8, edgecolor='none')
    ax.axhline(0, color='k', lw=0.7)
    for period_label, years in list(DROUGHT_DEFS.items())[1:]:
        ax.axvspan(min(years)-0.5, max(years)+0.5, color=DROUGHT_COLORS[period_label], alpha=0.08, zorder=0)
        mean_anom = cl_df.loc[cl_df.index.isin(years), 'precip_ann_anom_pct'].mean()
        if not np.isnan(mean_anom):
            ax.hlines(mean_anom, min(years)-0.5, max(years)+0.5, color=DROUGHT_COLORS[period_label], lw=2, ls='--', zorder=5)
    ax.set_title(f'{cl} -- {REGION_LABELS[cl]}', fontsize=9, fontweight='bold')
    ax.set_ylabel('Precip anomaly (%)', fontsize=7)
    ax.set_xlim(1999.5, 2019.5); ax.set_xticks(range(2000, 2020, 4))
    ax.tick_params(labelsize=7, axis='x', rotation=45)
    ax.grid(True, alpha=0.2, axis='y')

fig.legend(handles=[
    Patch(color=DROUGHT_COLORS['Drought 1 (2010-2019)'], alpha=0.6, label='2010-2019'),
    Patch(color=DROUGHT_COLORS['Drought 2 (2013-2019)'], alpha=0.6, label='2013-2019'),
    Patch(color=DROUGHT_COLORS['Drought 3 (2015-2019)'], alpha=0.6, label='2015-2019'),
], loc='lower center', ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.02), framealpha=0.9)

fig.suptitle('Annual precipitation anomaly vs 2000-2009 baseline -- by cluster', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/precip_anomaly_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

## III. Snow drought diagnosis (OGGM-based)

Uses OGGM's `snowfall_on_glacier_monthly` (actual simulated solid
precipitation at glacier elevation) rather than a basin-average
precip x fixed snow-fraction proxy. Framework follows Harpold et al.
(2017). Computed once (cluster/glacier level, not basin-set dependent).

In [ ]:
print('Loading OGGM snowfall on glacier for all clusters...')

import xarray as xr

snowfall_records = []

for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    for cl in CLUSTERS:
        path = nc_path(ds_name, cl)
        if not os.path.exists(path):
            continue
        try:
            ds = xr.open_dataset(path)
            if 'snowfall_on_glacier_monthly' not in ds:
                continue
            snow = ds['snowfall_on_glacier_monthly'].values
            annual_snow = np.nansum(np.nansum(snow, axis=1), axis=-1)
            for i, year in enumerate(range(2000, 2020)):
                snowfall_records.append({'dataset': ds_name, 'cluster': cl, 'year': year,
                                         'snowfall_kg': annual_snow[i + YEAR_OFFSET]})
            ds.close()
        except Exception as e:
            print(f'  Error {ds_name} {cl}: {e}')

df_snowfall = pd.DataFrame(snowfall_records)
print(f'Snowfall records: {len(df_snowfall):,}')

print()
print('Snow drought classification -- Harpold et al. (2017) framework')
print(f'{"Cluster":6s} {"Dataset":8s} {"Period":25s} {"Snow anom%":11s} {"P anom%":8s} {"T anom C":9s} {"Classification"}')
print('-' * 90)

snow_drought_records = []

for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    for cl in CLUSTERS:
        cl_snow = df_snowfall[(df_snowfall['dataset'] == ds_name) & (df_snowfall['cluster'] == cl)]
        if len(cl_snow) == 0:
            continue
        cl_clim = df_clim[df_clim['cluster'] == cl]
        snow_ts = cl_snow.set_index('year')['snowfall_kg']
        snow_base = snow_ts[snow_ts.index.isin(BASELINE)].mean()

        for period_label, years in list(DROUGHT_DEFS.items())[1:]:
            snow_drought = snow_ts[snow_ts.index.isin(years)].mean()
            snow_anom = ((snow_drought - snow_base) / snow_base * 100) if snow_base > 0 else np.nan
            winp_anom = (cl_clim[cl_clim['year'].isin(years)]['precip_win_anom_pct'].mean())
            temp_anom = (cl_clim[cl_clim['year'].isin(years)]['tmean_anom'].mean())

            snow_below, prec_below, temp_warm = snow_anom < -10, winp_anom < -15, temp_anom > 0.3

            if not snow_below:
                dtype = 'No snow drought'
            elif prec_below and temp_warm:
                dtype = 'Compound (dry + warm)'
            elif prec_below:
                dtype = 'Dry snow drought'
            elif temp_warm:
                dtype = 'Warm snow drought'
            else:
                dtype = 'Dry snow drought'

            print(f'{cl:6s} {ds_name:8s} {period_label:25s} {snow_anom:+11.1f}% {winp_anom:+8.1f}% {temp_anom:+9.2f}C {dtype}')
            snow_drought_records.append({'cluster': cl, 'dataset': ds_name, 'period': period_label,
                                         'snow_anom': snow_anom, 'winp_anom': winp_anom, 'temp_anom': temp_anom, 'dtype': dtype})
        print()

df_snow_drought = pd.DataFrame(snow_drought_records)

print('Consensus classification (majority vote across 3 datasets):')
for cl in CLUSTERS:
    for period_label in list(DROUGHT_DEFS.keys())[1:]:
        sub = df_snow_drought[(df_snow_drought['cluster'] == cl) & (df_snow_drought['period'] == period_label)]
        if len(sub) == 0:
            continue
        votes = sub['dtype'].value_counts()
        print(f'{cl:6s} {period_label:25s} {votes.index[0]} ({votes.iloc[0]}/3 datasets)')

## IV. Climate driver analysis

Compares temperature, precipitation, and PET anomalies during each drought period (all four definitions) to identify which driver dominates by cluster.

In [ ]:
driver_records = []
for cl in CLUSTERS:
    cl_df = df_clim[df_clim['cluster'] == cl].set_index('year')
    for period_label, years in DROUGHT_DEFS.items():
        period_df = cl_df[cl_df.index.isin(years)]
        driver_records.append({
            'cluster': cl, 'period': period_label,
            'precip_ann_anom': period_df['precip_ann_anom_pct'].mean(),
            'precip_win_anom': period_df['precip_win_anom_pct'].mean(),
            'tmean_anom':      period_df['tmean_anom'].mean(),
            'pet_anom_pct':    period_df['pet_anom_pct'].mean(),
        })

df_drivers = pd.DataFrame(driver_records)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
driver_vars = [
    ('precip_ann_anom', 'Annual precip anomaly (%)'), ('precip_win_anom', 'Winter precip anomaly (%)'),
    ('tmean_anom', 'Annual temp anomaly (C)'), ('pet_anom_pct', 'Annual PET anomaly (%)'),
]

x = np.arange(len(CLUSTERS))
width = 0.22
offsets = [-1.5*width, -0.5*width, 0.5*width, 1.5*width]

for ax, (var, ylabel) in zip(axes.flatten(), driver_vars):
    for i, (period_label, years) in enumerate(list(DROUGHT_DEFS.items())):
        vals = []
        for cl in CLUSTERS:
            sub = df_drivers[(df_drivers['cluster'] == cl) & (df_drivers['period'] == period_label)]
            vals.append(sub[var].values[0] if len(sub) else np.nan)
        ax.bar(x + offsets[i], vals, width, label=period_label, color=DROUGHT_COLORS[period_label], alpha=0.85, edgecolor='k', lw=0.3)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_xticks(x); ax.set_xticklabels(CLUSTERS, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=7, ncol=2)

fig.suptitle('Climate drivers during drought periods vs 2000-2009 baseline, by cluster', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/climate_drivers_by_period_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

## Becca's ideas: recession coefficient, routing, and aridity

## Addressing Becca's ideas: recession coefficient & routing

Becca's suggestion: estimate a recession coefficient from the falling
limb of the CAMELS hydrograph for each basin, to (1) improve estimates
of how much water actually reaches streams, and (2) understand basin
storage dynamics -- how long water takes to move through other stores.
Also: contextualize glacier importance against basin aridity.

In [ ]:
print('Loading daily streamflow and precipitation...')
q_day    = pd.read_csv(f'{CAMELS_FULL}/q_m3s_day.csv', parse_dates=['date'])
prec_day = pd.read_csv(f'{CAMELS_FULL}/precip_cr2met_mm_day.csv', parse_dates=['date'])

q_day    = q_day[(q_day['year'] >= 2000) & (q_day['year'] <= 2019)].reset_index(drop=True)
prec_day = prec_day[(prec_day['year'] >= 2000) & (prec_day['year'] <= 2019)].reset_index(drop=True)

print(f'Q daily:    {q_day.shape}')
print(f'Prec daily: {prec_day.shape}')

MIN_RECESSION_DAYS = 5
PREC_THRESHOLD_MM  = 1.0
MIN_Q_M3S          = 0.01

def exp_decay(t, q0, k):
    return q0 * np.exp(-k * t)

def compute_recession_k(gauge_id, q_series, prec_series):
    if gauge_id not in q_series.columns:
        return []
    if gauge_id not in prec_series.columns:
        prec = pd.Series(0, index=q_series.index)
    else:
        prec = prec_series[gauge_id]

    q = q_series[gauge_id].values
    p = prec.values
    k_vals = []

    i = 0
    while i < len(q) - MIN_RECESSION_DAYS:
        if (not np.isnan(q[i]) and q[i] > MIN_Q_M3S and
                not np.isnan(q[i+1]) and q[i+1] < q[i]):
            j = i + 1
            while (j < len(q) and not np.isnan(q[j]) and q[j] < q[j-1] and
                   q[j] > MIN_Q_M3S and (np.isnan(p[j]) or p[j] < PREC_THRESHOLD_MM)):
                j += 1
            seg_len = j - i
            if seg_len >= MIN_RECESSION_DAYS:
                t_seg = np.arange(seg_len, dtype=float)
                q_seg = q[i:j]
                try:
                    popt, _ = curve_fit(exp_decay, t_seg, q_seg, p0=[q_seg[0], 0.05],
                                        bounds=([0, 1e-6], [1e6, 2.0]), maxfev=2000)
                    k_vals.append(popt[1])
                except Exception:
                    pass
            i = j
        else:
            i += 1
    return k_vals

print('Computing recession coefficients for all gauges...')

recession_results = []
for gauge_id in GAUGE_INFO_FRAC:
    gid = str(gauge_id)
    k_vals = compute_recession_k(gid, q_day, prec_day)
    if len(k_vals) == 0:
        continue

    info = GAUGE_INFO_FRAC[gauge_id]
    recession_results.append({
        'gauge_id': gauge_id, 'gauge_name': info['gauge_name'][:35],
        'lat': info['cen_lat'], 'cluster': info['clusters'][0],
        'glacier_area_km2': info['glacier_area_km2'],
        'basin_area_km2': info.get('basin_area_km2', np.nan),
        'glac_pct': info['glacier_area_km2'] / info.get('basin_area_km2', np.nan) * 100
                    if info.get('basin_area_km2') else np.nan,
        'k_median': np.median(k_vals), 'k_mean': np.mean(k_vals),
        'k_std': np.std(k_vals), 'n_recession_segs': len(k_vals),
    })

df_recession = pd.DataFrame(recession_results)
print(f'Recession coefficients computed: {len(df_recession)} gauges')
print(f'Median k across all gauges: {df_recession["k_median"].median():.4f} day-1')
print('Median k by cluster:')
print(df_recession.groupby('cluster')['k_median'].median().round(4).to_string())

In [ ]:
attrs = pd.read_csv(f'{CAMELS_FULL}/catchment_attributes.csv', dtype={'gauge_id': str})
proxy_cols = ['gauge_id', 'baseflow_index_1979_2010', 'slope_fdc_1979_2010',
              'Q5_1979_2010', 'Q95_1979_2010', 'hfd_mean']
attrs_sub = attrs[proxy_cols].copy()

df_rec_full = df_recession.merge(attrs_sub, on='gauge_id', how='left')

print('Correlations between computed k and CAMELS-CL proxy metrics:')
print()
proxy_vars = {
    'baseflow_index_1979_2010': 'Baseflow index (expect: k neg corr)',
    'slope_fdc_1979_2010':      'Slope FDC (expect: k pos corr)',
    'Q5_1979_2010':             'Q5 low flow (expect: k neg corr)',
    'hfd_mean':                 'Half-flow date (expect: k neg corr)',
}

for col, label in proxy_vars.items():
    sub = df_rec_full[['k_median', col]].dropna()
    if len(sub) < 5:
        continue
    r, p = stats.pearsonr(sub['k_median'], sub[col])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  k vs {label[:35]:35s}: r={r:+.3f}  p={p:.3e}  {sig}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes_flat = axes.flatten()

cluster_colors_map = {
    'OT3':'#e6194b', 'DA1':'#f58231', 'DA2':'#ffe119', 'DA3':'#bfef45',
    'WA1':'#3cb44b', 'WA2':'#42d4f4', 'WA3':'#4363d8', 'WA4':'#911eb4', 'WA6':'#a9a9a9'
}

for ax, (col, label) in zip(axes_flat, proxy_vars.items()):
    for cl in CLUSTERS:
        grp = df_rec_full[df_rec_full['cluster'] == cl]
        if len(grp) == 0:
            continue
        ax.scatter(grp[col], grp['k_median'], c=cluster_colors_map.get(cl, '#888888'),
                   s=40, alpha=0.7, edgecolors='k', lw=0.3, label=cl, zorder=3)

    sub = df_rec_full[['k_median', col]].dropna()
    if len(sub) > 5:
        slope, intercept, r, p, _ = stats.linregress(sub[col], sub['k_median'])
        xfit = np.linspace(sub[col].min(), sub[col].max(), 100)
        ax.plot(xfit, slope*xfit + intercept, 'k--', lw=1.5, label=f'r={r:.2f} p={p:.3f}')

    ax.set_xlabel(label.split('(')[0].strip(), fontsize=9)
    ax.set_ylabel('Recession coefficient k (day-1)', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

fig.suptitle('Recession coefficient k vs CAMELS-CL proxy metrics\nColored by cluster  |  Dashed line = OLS fit',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/recession_k_vs_proxies.png', dpi=150, bbox_inches='tight')
plt.show()

print('Recession coefficient and proxy metrics by cluster:')
print(f'{"Cluster":6s} {"k_median":10s} {"BFI":8s} {"slope_fdc":10s} {"hfd_mean":10s} {"n"}')
print('-' * 55)
for cl in CLUSTERS:
    grp = df_rec_full[df_rec_full['cluster'] == cl]
    if len(grp) == 0:
        continue
    print(f'{cl:6s} {grp["k_median"].median():10.4f} '
          f'{grp["baseflow_index_1979_2010"].median():8.3f} '
          f'{grp["slope_fdc_1979_2010"].median():10.3f} '
          f'{grp["hfd_mean"].median():10.1f} {len(grp):3d}')

**Note:** the regression cell below reads `per_basin_regression_CR2MET.csv` from `04_regression_analysis.ipynb`'s `REG_DIR` (columns `b_glacier`, `ci_lo_glacier`, `ci_hi_glacier`, `sig`, `r2`). If the file is not there, the cell prints a message and skips the regression panels. The lag panel is skipped unless a `lag_results_CR2MET.csv` exists in `REG_DIR/lag_analysis`.

In [ ]:
pred   = 'b_glacier'
lo_col = 'ci_lo_glacier'
hi_col = 'ci_hi_glacier'

reg_path = f'{REG_DIR}/per_basin_regression_CR2MET.csv'
if os.path.exists(reg_path):
    df_reg = pd.read_csv(reg_path, dtype={'gauge_id': str})
    df_reg['sig'] = ((df_reg[lo_col] > 0) | (df_reg[hi_col] < 0))
    print(f'Regression results loaded: {len(df_reg)} basins')
else:
    print(f'Regression CSV not found at {reg_path}')
    print('Run 04_regression_analysis.ipynb first')
    df_reg = pd.DataFrame(columns=['gauge_id', pred, lo_col, hi_col, 'sig', 'r2'])

lag_path = f'{REG_DIR}/lag_analysis/lag_results_CR2MET.csv'
if os.path.exists(lag_path):
    df_lag  = pd.read_csv(lag_path, dtype={'gauge_id': str})
    has_lag = True
    print(f'Lag results loaded: {len(df_lag)} gauges')
else:
    has_lag = False
    print('Lag results not found -- skipping lag panel')

df_k_reg = df_rec_full.merge(
    df_reg[['gauge_id', pred, lo_col, hi_col, 'sig', 'r2']], on='gauge_id', how='left')

if has_lag:
    df_k_reg = df_k_reg.merge(
        df_lag[['gauge_id', 'best_lag_months', 'best_r']], on='gauge_id', how='left')

print('Key correlations with recession coefficient k:')
print()

test_vars = [
    (pred,               'beta_glacier (regression coef)'),
    ('r2',               'Per-basin R2'),
    ('glac_pct',         'Glacier % of basin'),
    ('glacier_area_km2', 'Glacier area (km2)'),
]
if has_lag:
    test_vars.append(('best_lag_months', 'Best lag (months)'))

for col, label in test_vars:
    sub = df_k_reg[['k_median', col]].dropna()
    if len(sub) < 5:
        continue
    r, p = stats.pearsonr(sub['k_median'], sub[col])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  k vs {label:35s}: r={r:+.3f}  p={p:.3e}  {sig}')

n_panels = 4 if has_lag else 3
fig, axes = plt.subplots(1, n_panels, figsize=(5*n_panels, 5))

ax = axes[0]
sig_mask = df_k_reg['sig'] == True
ax.scatter(df_k_reg.loc[~sig_mask, 'k_median'], df_k_reg.loc[~sig_mask, pred],
           c='#cc0000', s=30, alpha=0.6, edgecolors='none', label='Not significant')
ax.scatter(df_k_reg.loc[sig_mask, 'k_median'], df_k_reg.loc[sig_mask, pred],
           c='#0044cc', s=50, alpha=0.8, edgecolors='k', lw=0.4, label='Significant')
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.set_xlabel('Recession coefficient k (day-1)', fontsize=10)
ax.set_ylabel('beta_glacier (standardised)', fontsize=10)
ax.set_title('k vs glacier regression coefficient\nFaster drainage = stronger glacier signal?', fontsize=9)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

sub = df_k_reg[[pred, 'k_median']].dropna()
if len(sub) > 5:
    slope, intercept, r, p, _ = stats.linregress(sub['k_median'], sub[pred])
    xfit = np.linspace(sub['k_median'].min(), sub['k_median'].max(), 100)
    ax.plot(xfit, slope*xfit + intercept, 'k--', lw=1.2, label=f'r={r:.2f}')
    ax.legend(fontsize=8)

ax = axes[1]
for cl in CLUSTERS:
    grp = df_k_reg[df_k_reg['cluster'] == cl].dropna(subset=['k_median','r2'])
    if len(grp) == 0:
        continue
    ax.scatter(grp['k_median'], grp['r2'], c=cluster_colors_map.get(cl, '#888'),
               s=35, alpha=0.7, edgecolors='k', lw=0.3, label=cl)
ax.set_xlabel('Recession coefficient k (day-1)', fontsize=10)
ax.set_ylabel('Per-basin R2', fontsize=10)
ax.set_title('k vs regression R2\nSlower recession = lower R2?', fontsize=9)
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

ax = axes[2]
for cl in CLUSTERS:
    grp = df_k_reg[df_k_reg['cluster'] == cl].dropna(subset=['k_median','glac_pct'])
    if len(grp) == 0:
        continue
    ax.scatter(grp['k_median'], grp['glac_pct'], c=cluster_colors_map.get(cl, '#888'),
               s=35, alpha=0.7, edgecolors='k', lw=0.3, label=cl)
ax.set_xlabel('Recession coefficient k (day-1)', fontsize=10)
ax.set_ylabel('Glacier % of basin', fontsize=10)
ax.set_title('k vs glacierization\nMore glacierized = different drainage?', fontsize=9)
ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

if has_lag:
    ax = axes[3]
    for cl in CLUSTERS:
        grp = df_k_reg[df_k_reg['cluster'] == cl].dropna(subset=['k_median','best_lag_months'])
        if len(grp) == 0:
            continue
        ax.scatter(grp['k_median'], grp['best_lag_months'], c=cluster_colors_map.get(cl, '#888'),
                   s=35, alpha=0.7, edgecolors='k', lw=0.3, label=cl)
    ax.set_xlabel('Recession coefficient k (day-1)', fontsize=10)
    ax.set_ylabel('Best lag (months)', fontsize=10)
    ax.set_title('k vs glacier-to-Q lag\nSlower recession = longer lag?', fontsize=9)
    ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.3)

    sub = df_k_reg[['k_median','best_lag_months']].dropna()
    if len(sub) > 5:
        slope, intercept, r, p, _ = stats.linregress(sub['k_median'], sub['best_lag_months'])
        xfit = np.linspace(sub['k_median'].min(), sub['k_median'].max(), 100)
        axes[3].plot(xfit, slope*xfit + intercept, 'k--', lw=1.2, label=f'r={r:.2f}')
        axes[3].legend(fontsize=8)

fig.suptitle(
    'Recession coefficient k vs glacier signal strength and routing\n'
    'Tests whether basin storage dynamics control glacier melt detectability',
    fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/recession_k_vs_glacier_signal.png', dpi=150, bbox_inches='tight')
plt.show()

df_k_reg.to_csv(f'{DROUGHT_DIR}/recession_coefficients_all_basins.csv', index=False)
print(f'Saved: {DROUGHT_DIR}/recession_coefficients_all_basins.csv')

## Becca's routing idea: route glacier melt through basin storage before computing fraction

Applies each basin's recession coefficient k to shift glacier melt
forward in time (exponential decay kernel) before dividing by observed
Q. **Framing note (Becca):** this does NOT imply all glacier melt
reaches streamflow. Storage, evaporation, and deep groundwater loss are
not accounted for -- the routed fraction is a timing index, not a water
balance closure.

In [ ]:
rec_path = f'{DROUGHT_DIR}/recession_coefficients_all_basins.csv'
if os.path.exists(rec_path):
    df_rec = pd.read_csv(rec_path, dtype={'gauge_id': str})
    k_by_gauge = df_rec.groupby('gauge_id')['k_median'].median().to_dict()
    print(f'Recession coefficients loaded: {len(k_by_gauge)} basins')
else:
    k_by_gauge = {}
    print(f'WARNING: recession coefficients CSV not found at {rec_path}')

K_CLUSTER_MEDIANS = {'OT3': 0.025, 'DA1': 0.037, 'DA2': 0.055, 'DA3': 0.068,
                     'WA1': 0.055, 'WA2': 0.062, 'WA3': 0.050, 'WA4': 0.079, 'WA6': 0.132}

print('Computing routed glacier fractions for all basins...')

routed_records = []

for gauge_id, info in GAUGE_INFO_FRAC.items():
    gid_str = str(gauge_id)
    k = k_by_gauge.get(gid_str, None)
    if k is None:
        cl = info['clusters'][0]
        k = K_CLUSTER_MEDIANS.get(cl, 0.05)
        k_source = 'cluster_median'
    else:
        k_source = 'basin_specific'

    for ds_name in ['CR2MET', 'ERA5', 'CRU']:
        res = RESULTS[ds_name].get(gauge_id)
        if res is None:
            continue
        m  = res['monthly']
        cy = res.get('complete_years', YEARS)
        mc = (m[m['year'].isin(cy)].sort_values(['year','month']).reset_index(drop=True))

        if len(mc) < 12:
            continue

        glac = mc['glacier_runoff_m3'].values.astype(float)
        q    = mc['q_m3_mon'].values.astype(float)
        n    = len(glac)

        max_lag = 12
        lags = np.arange(0, max_lag + 1)
        weights = np.exp(-k * lags * 30)
        weights = weights / weights.sum()

        routed = np.zeros(n)
        for i in range(n):
            for lag, w in enumerate(weights):
                if i + lag < n:
                    routed[i + lag] += glac[i] * w

        mc['routed_glac_m3'] = routed

        ann = mc.groupby('year').agg(raw_glac=('glacier_runoff_m3', 'sum'),
                                     routed_glac=('routed_glac_m3', 'sum'),
                                     q_tot=('q_m3_mon', 'sum')).reset_index()
        ann = ann[ann['q_tot'] > 0]
        if len(ann) == 0:
            continue

        ann_raw_frac    = (ann['raw_glac']    / ann['q_tot']).mean() * 100
        ann_routed_frac = (ann['routed_glac'] / ann['q_tot']).mean() * 100

        mc_mar = mc[mc['month'] == 3]
        mc_mar = mc_mar[mc_mar['q_m3_mon'] > 0]
        mar_raw_frac    = (mc_mar['glacier_runoff_m3'] / mc_mar['q_m3_mon']).mean()*100 if len(mc_mar) else np.nan
        mar_routed_frac = (mc_mar['routed_glac_m3']    / mc_mar['q_m3_mon']).mean()*100 if len(mc_mar) else np.nan

        routed_records.append({
            'gauge_id': gauge_id, 'gauge_name': info['gauge_name'][:35], 'lat': info['cen_lat'],
            'cluster': info['clusters'][0], 'glacier_area_km2': info['glacier_area_km2'],
            'basin_area_km2': info.get('basin_area_km2', np.nan), 'dataset': ds_name,
            'k': k, 'k_source': k_source,
            'ann_raw_frac': ann_raw_frac, 'ann_routed_frac': ann_routed_frac,
            'ann_diff': ann_routed_frac - ann_raw_frac,
            'mar_raw_frac': mar_raw_frac, 'mar_routed_frac': mar_routed_frac,
            'mar_diff': mar_routed_frac - mar_raw_frac if mar_routed_frac is not None else np.nan,
        })

df_routed = pd.DataFrame(routed_records)
print(f'Routed fraction records: {len(df_routed):,}')

print()
print('Impact of routing on glacier fraction (CR2MET, cluster means):')
print(f'{"Cluster":6s} {"k":6s} {"Ann raw%":9s} {"Ann rout%":10s} {"Diff":6s} {"Mar raw%":9s} {"Mar rout%":10s} {"Diff"}')
print('-' * 75)

cr2 = df_routed[df_routed['dataset'] == 'CR2MET']
for cl in CLUSTERS:
    sub = cr2[cr2['cluster'] == cl]
    if len(sub) == 0:
        continue
    k_med = sub['k'].median()
    print(f'{cl:6s} {k_med:.3f} {sub["ann_raw_frac"].mean():9.2f} {sub["ann_routed_frac"].mean():10.2f} '
          f'{sub["ann_diff"].mean():+6.2f} {sub["mar_raw_frac"].mean():9.2f} '
          f'{sub["mar_routed_frac"].mean():10.2f} {sub["mar_diff"].mean():+6.2f}')

df_routed.to_csv(f'{DROUGHT_DIR}/glacier_fractions_routed.csv', index=False)
print(f'Saved: {DROUGHT_DIR}/glacier_fractions_routed.csv')
print()
print('FRAMING NOTE (Becca): routed fractions improve the TIMING of the comparison')
print('only. They do NOT imply all glacier melt reaches the stream -- storage,')
print('deep groundwater loss, and evaporation are not accounted for.')

## Spatial map: annual fraction, drought amplification, and recession k

Three-panel Chile-wide map (annual glacier fraction, drought
amplification factor, recession coefficient k), plus a table of the
20 most drought-amplified basins nationally.

In [ ]:
from matplotlib.colors import Normalize, to_hex
from matplotlib.cm import ScalarMappable

if 'bounds' not in dir():
    bounds = gpd.read_file(BOUNDS_PATH)
    bounds['gauge_id'] = bounds['gauge_id'].astype(str)

if 'chile' not in dir():
    try:
        world = gpd.read_file('https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip')
        chile = world[world['NAME'] == 'Chile']
        arg   = world[world['NAME'] == 'Argentina']
        has_borders = True
    except Exception:
        has_borders = False

frac_by_gauge = {}
for gauge_id, res in RESULTS['CR2MET'].items():
    if res is None or gauge_id not in GAUGE_INFO_FRAC:
        continue
    m  = res['monthly']
    cy = res.get('complete_years', YEARS)
    ann = (m[m['year'].isin(cy)].groupby('year')
           .agg(glac=('glacier_runoff_m3','sum'), q=('q_m3_mon','sum')))
    ann = ann[ann['q'] > 0]
    if len(ann):
        frac_by_gauge[gauge_id] = (ann['glac']/ann['q']).mean() * 100

amp_by_gauge = {}
for gauge_id, res in RESULTS['CR2MET'].items():
    if res is None:
        continue
    info = GAUGE_INFO_FRAC.get(gauge_id)
    if info is None:
        continue
    m  = res['monthly']
    cy = res.get('complete_years', YEARS)

    def period_frac(years):
        valid = [y for y in years if y in cy]
        if len(valid) < 2:
            return np.nan
        sub = m[m['year'].isin(valid)]
        ann = sub.groupby('year').agg(glac=('glacier_runoff_m3','sum'), q=('q_m3_mon','sum'))
        ann = ann[ann['q'] > 0]
        return (ann['glac']/ann['q']).mean() if len(ann) else np.nan

    f_base    = period_frac(BASELINE)
    f_drought = period_frac(list(range(2010, 2020)))
    if not any(np.isnan([f_base, f_drought])) and f_base > 0:
        amp_by_gauge[gauge_id] = f_drought / f_base

bounds_plot = bounds.copy()
bounds_plot['frac_ann']   = bounds_plot['gauge_id'].map(frac_by_gauge)
bounds_plot['amp_factor'] = bounds_plot['gauge_id'].map(amp_by_gauge)
bounds_plot['k_median']   = bounds_plot['gauge_id'].map(
    df_recession.set_index('gauge_id')['k_median'].to_dict())

fig, axes = plt.subplots(1, 3, figsize=(18, 22))

map_configs = [
    ('frac_ann', plt.cm.Blues, Normalize(0, 30),
     'Annual glacier fraction (%)', 'Mean annual glacier runoff / Q\n(CR2MET, 2000-2019)'),
    ('amp_factor', plt.cm.YlOrRd, Normalize(1, 4),
     'Drought amplification factor\n(2010-2019 / 2000-2009)', 'Glacier fraction amplification\nduring megadrought (CR2MET)'),
    ('k_median', plt.cm.RdYlBu, Normalize(0.00, 0.15),
     'Recession coefficient k (day-1)\nRed = slow drainage | Blue = fast',
     'Basin drainage rate\nRed = high storage | Blue = fast runoff'),
]

for ax, (col, cmap, norm, cbar_label, title) in zip(axes, map_configs):
    if has_borders:
        chile.plot(ax=ax, facecolor='#f2f2f2', edgecolor='#999999', linewidth=0.5, zorder=0)
        arg.plot(ax=ax, facecolor='#fafafa', edgecolor='#999999', linewidth=0.5, zorder=0)

    bounds.plot(ax=ax, facecolor='#e0e0e0', edgecolor='#cccccc', linewidth=0.2, zorder=1)

    face_colors = [to_hex(cmap(norm(v))) if (v is not None and not np.isnan(v)) else '#e0e0e0'
                   for v in bounds_plot[col].values]
    bounds_plot.plot(ax=ax, color=face_colors, edgecolor='#555555', linewidth=0.3, zorder=2)

    ax.set_xlim(-76, -64); ax.set_ylim(-56, -17); ax.set_aspect('equal')
    ax.tick_params(labelsize=7); ax.grid(True, alpha=0.15)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=8)

    n_valid = bounds_plot[col].notna().sum()
    ax.text(0.03, 0.02, f'n={n_valid}', transform=ax.transAxes, fontsize=8)

    sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02)
    cb.set_label(cbar_label, fontsize=8); cb.ax.tick_params(labelsize=7)

fig.suptitle(
    'Spatial analysis: where and why glacier melt matters\n'
    'Left: annual contribution  |  Centre: drought amplification  |  Right: basin drainage dynamics',
    fontsize=11, y=1.002)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/spatial_analysis_three_maps.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 20 most drought-amplified basins (CR2MET, 2010-2019 vs baseline):')
top_amp = (pd.DataFrame({'gauge_id': list(amp_by_gauge.keys()), 'amp_factor': list(amp_by_gauge.values())})
           .sort_values('amp_factor', ascending=False).head(20))

for _, row in top_amp.iterrows():
    gid  = row['gauge_id']
    info = GAUGE_INFO_FRAC.get(gid, {})
    frac = frac_by_gauge.get(gid, np.nan)
    print(f'  {gid:10s} {info.get("gauge_name","")[:35]:35s} '
          f'amp={row["amp_factor"]:.2f}x  ann_frac={frac:.1f}%  '
          f'cluster={info.get("clusters",["?"])[0]}  lat={info.get("cen_lat",np.nan):.1f}S')

## Basin aridity analysis (Becca)

Aridity index (PET/P) characterizes how water-limited a basin is. In
more arid basins, glaciers should be proportionally more important as a
water source since there are fewer alternatives. Uses
`aridity_cr2met_1979_2010` from CAMELS-CL catchment attributes.

In [ ]:
attrs = pd.read_csv(f'{CAMELS_FULL}/catchment_attributes.csv', dtype={'gauge_id': str})

attr_cols = ['gauge_id', 'gauge_lat', 'gauge_lon',
             'aridity_cr2met_1979_2010', 'aridity_cr2met_1990_2010',
             'mean_elev', 'med_elev', 'max_elev', 'outlet_camels_elev', 'mean_slope_perc',
             'baseflow_index_1979_2010', 'frac_snow_cr2met_1979_2010',
             'p_mean_cr2met_1979_2010', 'pet_mean_1979_2010', 'q_mean_1979_2010',
             'runoff_ratio_cr2met_1979_2010']
attrs_sub = attrs[[c for c in attr_cols if c in attrs.columns]].copy()

frac_records = []
for gauge_id, res in RESULTS['CR2MET'].items():
    if res is None:
        continue
    info = GAUGE_INFO_FRAC.get(gauge_id)
    if info is None:
        continue
    m  = res['monthly']
    cy = res.get('complete_years', YEARS)

    ann = (m[m['year'].isin(cy)].groupby('year')
           .agg(glac=('glacier_runoff_m3','sum'), q=('q_m3_mon','sum')))
    ann = ann[ann['q'] > 0]
    if len(ann) == 0:
        continue

    f_base    = (ann.loc[ann.index.isin(BASELINE), 'glac'] / ann.loc[ann.index.isin(BASELINE), 'q']).mean()
    f_drought = (ann.loc[ann.index.isin(range(2010,2020)), 'glac'] / ann.loc[ann.index.isin(range(2010,2020)), 'q']).mean()
    amp = f_drought / f_base if (f_base > 0 and not np.isnan(f_base)) else np.nan

    feb = m[(m['month'] == 2) & m['year'].isin(cy)]
    feb = feb[feb['q_m3_mon'] > 0]
    feb_frac = (feb['glacier_runoff_m3'] / feb['q_m3_mon']).mean() * 100 if len(feb) else np.nan

    frac_records.append({
        'gauge_id': gauge_id, 'gauge_name': info['gauge_name'][:35], 'lat': info['cen_lat'],
        'cluster': info['clusters'][0], 'glacier_area_km2': info['glacier_area_km2'],
        'basin_area_km2': info.get('basin_area_km2', np.nan),
        'glac_pct': info['glacier_area_km2'] / info.get('basin_area_km2', np.nan) * 100
                    if info.get('basin_area_km2') else np.nan,
        'mean_ann_frac': (ann['glac'] / ann['q']).mean() * 100,
        'feb_frac': feb_frac, 'amp_factor': amp,
    })

df_aridity = pd.DataFrame(frac_records).merge(attrs_sub, on='gauge_id', how='left')

pred, lo_col, hi_col = 'b_glacier', 'ci_lo_glacier', 'ci_hi_glacier'
reg_path = f'{REG_DIR}/per_basin_regression_CR2MET.csv'
if os.path.exists(reg_path):
    df_cr2_sig = pd.read_csv(reg_path, dtype={'gauge_id': str})
    df_cr2_sig['sig'] = ((df_cr2_sig[lo_col] > 0) | (df_cr2_sig[hi_col] < 0))
    df_aridity = df_aridity.merge(
        df_cr2_sig[['gauge_id','sig', pred, 'r2']].rename(columns={pred: 'beta_glacier'}),
        on='gauge_id', how='left')
else:
    print(f'Regression CSV not found at {reg_path} -- sig/beta columns will be NaN')
    df_aridity['sig'] = np.nan
    df_aridity['beta_glacier'] = np.nan
    df_aridity['r2'] = np.nan

print(f'Aridity analysis dataset: {len(df_aridity)} basins')
print('Aridity index by cluster (CR2MET 1979-2010):')
print(df_aridity.groupby('cluster')['aridity_cr2met_1979_2010'].agg(['median','min','max']).round(2).to_string())

cluster_colors_map = {
    'OT3':'#e6194b', 'DA1':'#f58231', 'DA2':'#ffe119', 'DA3':'#bfef45',
    'WA1':'#3cb44b', 'WA2':'#42d4f4', 'WA3':'#4363d8', 'WA4':'#911eb4', 'WA6':'#a9a9a9'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

plot_configs = [
    ('mean_ann_frac', 'Mean annual glacier fraction (%)', 'Annual glacier contribution vs basin aridity'),
    ('feb_frac', 'February glacier fraction (%)', 'Peak summer (February) glacier contribution vs aridity'),
    ('amp_factor', 'Drought amplification factor\n(2010-2019 / 2000-2009)', 'Drought amplification vs aridity'),
]

for ax, (ycol, ylabel, title) in zip(axes, plot_configs):
    sub = df_aridity[['aridity_cr2met_1979_2010', ycol, 'cluster', 'sig']].dropna()

    ns = sub[sub['sig'] == False]
    ax.scatter(ns['aridity_cr2met_1979_2010'], ns[ycol],
               c=[cluster_colors_map.get(c,'#888') for c in ns['cluster']],
               s=30, alpha=0.4, edgecolors='none', zorder=2)

    sg = sub[sub['sig'] == True]
    ax.scatter(sg['aridity_cr2met_1979_2010'], sg[ycol],
               c=[cluster_colors_map.get(c,'#888') for c in sg['cluster']],
               s=70, alpha=0.9, edgecolors='k', lw=0.5, zorder=3, label='Significant glacier signal')

    ax.set_xscale('log')
    ax.axvline(1.0, color='k', lw=0.8, ls='--', alpha=0.5, label='Aridity = 1 (arid/humid boundary)')

    x = np.log10(sub['aridity_cr2met_1979_2010'].clip(lower=0.1))
    y = sub[ycol]
    if len(x) > 5:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit_log = np.linspace(x.min(), x.max(), 100)
        xfit = 10**xfit_log
        ax.plot(xfit, slope*xfit_log + intercept, 'k-', lw=1.5, alpha=0.7, label=f'r={r:.2f} (log aridity)')

    ax.set_xlabel('Aridity index (PET/P) -- log scale', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax.text(0.25, 0.97, 'Humid', transform=ax.transAxes, fontsize=8, va='top', color='#2980b9', alpha=0.7)
    ax.text(0.75, 0.97, 'Arid', transform=ax.transAxes, fontsize=8, va='top', color='#e67e22', alpha=0.7)

handles = [plt.scatter([],[], c=cluster_colors_map.get(cl,'#888'), s=40, label=cl)
           for cl in CLUSTERS if cl in cluster_colors_map]
fig.legend(handles=handles, loc='lower center', ncol=len(handles), fontsize=8, bbox_to_anchor=(0.5, -0.02), framealpha=0.9)

fig.suptitle(
    'Glacier importance as a function of basin aridity\n'
    'Aridity index = PET/P  |  Large points = statistically significant glacier signal  |  Colored by cluster',
    fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/aridity_vs_glacier_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Correlation of aridity with glacier importance metrics:')
print()

test_pairs = [
    ('aridity_cr2met_1979_2010', 'mean_ann_frac', 'Aridity vs annual glacier fraction'),
    ('aridity_cr2met_1979_2010', 'feb_frac', 'Aridity vs February glacier fraction'),
    ('aridity_cr2met_1979_2010', 'amp_factor', 'Aridity vs drought amplification factor'),
    ('aridity_cr2met_1979_2010', 'beta_glacier', 'Aridity vs beta_glacier (regression coef)'),
]

for xcol, ycol, label in test_pairs:
    sub = df_aridity[[xcol, ycol]].dropna()
    if len(sub) < 5:
        continue
    r_raw, p_raw = stats.pearsonr(sub[xcol], sub[ycol])
    r_log, p_log = stats.pearsonr(np.log10(sub[xcol].clip(lower=0.1)), sub[ycol])
    sig_raw = '***' if p_raw < 0.001 else '**' if p_raw < 0.01 else '*' if p_raw < 0.05 else 'ns'
    sig_log = '***' if p_log < 0.001 else '**' if p_log < 0.01 else '*' if p_log < 0.05 else 'ns'
    print(f'{label}:')
    print(f'  Raw aridity:     r={r_raw:+.3f}  p={p_raw:.3e}  {sig_raw}')
    print(f'  Log10 aridity:   r={r_log:+.3f}  p={p_log:.3e}  {sig_log}')
    print()

bins   = [0, 0.5, 1.0, 2.0, 5.0, 30.0]
labels = ['<0.5\n(very humid)', '0.5-1.0\n(humid)', '1.0-2.0\n(semi-arid)', '2.0-5.0\n(arid)', '>5.0\n(hyper-arid)']

df_aridity['aridity_class'] = pd.cut(df_aridity['aridity_cr2met_1979_2010'], bins=bins, labels=labels)

print('Median glacier fraction by aridity class:')
print(df_aridity.groupby('aridity_class', observed=True)[
    ['mean_ann_frac','feb_frac','amp_factor']].agg(['median','count']).round(2).to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (col, ylabel) in zip(axes, [
    ('mean_ann_frac', 'Annual glacier fraction (%)'),
    ('feb_frac', 'February glacier fraction (%)'),
    ('amp_factor', 'Drought amplification factor'),
]):
    groups = [df_aridity[df_aridity['aridity_class'] == lab][col].dropna() for lab in labels]
    bp = ax.boxplot(groups, patch_artist=True, medianprops=dict(color='k', lw=2))

    colors_box = ['#2980b9','#27ae60','#f39c12','#e67e22','#c0392b']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color); patch.set_alpha(0.7)

    ax.set_xticklabels(labels, fontsize=8)
    ax.set_xlabel('Aridity class (PET/P)', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    for i, grp in enumerate(groups):
        ax.text(i+1, ax.get_ylim()[0], f'n={len(grp)}', ha='center', fontsize=7, va='bottom', color='#444444')

fig.suptitle(
    'Glacier importance by aridity class (PET/P)\n'
    'Blue = very humid  |  Red = hyper-arid  |  Lines = median  |  Boxes = IQR',
    fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{DROUGHT_DIR}/glacier_importance_by_aridity_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Computing seasonal aridity by gauge...')

NON_GAUGE    = ['date','year','month','day']
gauge_cols_p = [c for c in precip_df.columns if c not in NON_GAUGE]
gauge_cols_e = [c for c in pet_df.columns    if c not in NON_GAUGE]

p_djf = (precip_df[precip_df['month'].isin([12,1,2])].groupby('year')[gauge_cols_p].sum(numeric_only=True).mean())
e_djf = (pet_df[pet_df['month'].isin([12,1,2])].groupby('year')[gauge_cols_e].sum(numeric_only=True).mean())
p_jja = (precip_df[precip_df['month'].isin([6,7,8])].groupby('year')[gauge_cols_p].sum(numeric_only=True).mean())
e_jja = (pet_df[pet_df['month'].isin([6,7,8])].groupby('year')[gauge_cols_e].sum(numeric_only=True).mean())

seasonal_records = []
for gauge_id in GAUGE_INFO_FRAC:
    gid = str(gauge_id)
    p_s, e_s = p_djf.get(gid, np.nan), e_djf.get(gid, np.nan)
    p_w, e_w = p_jja.get(gid, np.nan), e_jja.get(gid, np.nan)
    seasonal_records.append({
        'gauge_id': gauge_id,
        'aridity_summer': e_s/p_s if (p_s > 0 and not np.isnan(p_s)) else np.nan,
        'aridity_winter': e_w/p_w if (p_w > 0 and not np.isnan(p_w)) else np.nan,
        'precip_summer': p_s, 'precip_winter': p_w, 'pet_summer': e_s,
    })

df_seasonal = pd.DataFrame(seasonal_records)

for col in ['aridity_summer','aridity_winter','precip_summer','precip_winter','pet_summer']:
    if col in df_aridity.columns:
        df_aridity = df_aridity.drop(columns=[col])
df_aridity = df_aridity.merge(df_seasonal, on='gauge_id', how='left')
print(f'Seasonal aridity merged: {df_seasonal["aridity_summer"].notna().sum()} gauges')

print('Seasonal aridity by cluster:')
print(f'{"Cluster":6s} {"Ann aridity":12s} {"Summer aridity":15s} {"Winter aridity":15s} {"Summer precip(mm)"}')
print('-' * 65)
for cl in CLUSTERS:
    sub = df_aridity[df_aridity['cluster'] == cl]
    if len(sub) == 0:
        continue
    print(f'{cl:6s} {sub["aridity_cr2met_1979_2010"].median():12.2f} '
          f'{sub["aridity_summer"].median():15.2f} {sub["aridity_winter"].median():15.2f} '
          f'{sub["precip_summer"].median():15.1f}')

print()
print('Does summer aridity predict glacier fraction better than annual?')
for col, label in [('aridity_cr2met_1979_2010', 'Annual aridity'), ('aridity_summer', 'Summer aridity DJF')]:
    sub = df_aridity[[col, 'mean_ann_frac']].dropna()
    r, p_val = stats.pearsonr(np.log10(sub[col].clip(lower=0.01)), sub['mean_ann_frac'])
    print(f'  {label:25s}: r={r:+.3f}  p={p_val:.3e}')

### Long-term aridity trend (OGGM lista files, 1901/1960/1979-2019)

CAMELS-CL only covers 2000-2019 -- too short for trend detection. Uses
glacier-location temperature/precipitation from OGGM's `lista_temp.csv`
/ `lista_prec.csv` (CR2MET back to 1960, ERA5 to 1979, CRU to 1901) with
a Hamon PET proxy, to test whether summer aridity at glacier elevations
shows a genuine long-term trend.

**Result:** WA1 glacier-elevation summer warming is +0.9 to +1.2C since
1960, robust across all three datasets (temperature-driven, not
precipitation-driven -- summer precip shows no significant trend in any
dataset). Suggestive but not fully robust evidence of increasing summer
aridity in WA2 and Patagonia (WA3-WA4), particularly in CR2MET/CRU. WA1
itself shows no significant trend in the aridity *ratio*, despite the
clear warming, because both terms move together in a way that keeps the
ratio's trend within noise over this window.

In [ ]:
LISTA_START_YEARS = {'CR2MET': 1960, 'ERA5': 1979, 'CRU': 1901}

def hamon_pet_monthly(tmean_c):
    return np.maximum(0, 0.55 * tmean_c + 8)

def load_lista_cluster_mean(ds_name, cluster, variable='temp'):
    start_year = LISTA_START_YEARS[ds_name]
    fname = 'lista_temp.csv' if variable == 'temp' else 'lista_prec.csv'
    path = f'{OUTPUT_ROOT}/{ds_name}/{cluster}/{fname}'
    if not os.path.exists(path):
        return None

    df = pd.read_csv(path, header=0)
    n_rows = len(df)
    n_years = n_rows // 12

    years  = np.repeat(range(start_year, start_year + n_years), 12)[:n_rows]
    months = (list(range(1, 13)) * n_years)[:n_rows]

    glacier_cols = [c for c in df.columns]
    df['year'] = years
    df['month'] = months
    df['cluster_mean'] = df[glacier_cols].mean(axis=1)

    return df[['year', 'month', 'cluster_mean']]

print('Computing long-term summer aridity proxy by cluster...')
lt_records = []

for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    for cl in CLUSTERS:
        gauges_in_cl = [gid for gid, info in GAUGE_INFO_FRAC.items() if info['clusters'][0] == cl]
        if not gauges_in_cl:
            continue

        t_df = load_lista_cluster_mean(ds_name, cl, 'temp')
        p_df = load_lista_cluster_mean(ds_name, cl, 'prec')
        if t_df is None or p_df is None:
            continue

        merged = t_df.merge(p_df, on=['year','month'], suffixes=('_temp','_prec'))

        summer = merged[merged['month'].isin([12, 1, 2])].copy()
        summer['pet_proxy'] = hamon_pet_monthly(summer['cluster_mean_temp'])

        ann_summer = summer.groupby('year').agg(
            precip_sum=('cluster_mean_prec', 'sum'), pet_sum=('pet_proxy', 'sum'),
            tmean=('cluster_mean_temp', 'mean')).reset_index()

        ann_summer['aridity_proxy'] = (ann_summer['pet_sum'] /
                                       ann_summer['precip_sum'].where(ann_summer['precip_sum'] > 0))

        for _, row in ann_summer.iterrows():
            lt_records.append({
                'dataset': ds_name, 'cluster': cl, 'year': int(row['year']),
                'aridity_proxy': row['aridity_proxy'], 'precip_sum': row['precip_sum'],
                'pet_sum': row['pet_sum'], 'tmean': row['tmean'],
            })

df_lt = pd.DataFrame(lt_records)
df_lt = df_lt[~((df_lt['dataset'] == 'CRU')    & (df_lt['year'] > 2019))]
df_lt = df_lt[~((df_lt['dataset'] == 'CR2MET') & (df_lt['year'] > 2019))]

print(f'Long-term records: {len(df_lt):,}')
print('Year ranges by dataset:')
for ds in ['CR2MET', 'ERA5', 'CRU']:
    sub = df_lt[df_lt['dataset'] == ds]
    if len(sub):
        print(f'  {ds}: {int(sub["year"].min())} to {int(sub["year"].max())}')

print()
print('Long-term summer aridity trend by cluster and dataset:')
print(f'{"Cluster":6s} {"Dataset":8s} {"Period":15s} {"Slope/yr":10s} {"r":6s} {"p":8s} {"Sig"}')
print('-' * 65)

for cl in CLUSTERS:
    for ds_name in ['CR2MET', 'ERA5', 'CRU']:
        sub = (df_lt[(df_lt['cluster'] == cl) & (df_lt['dataset'] == ds_name)]
               .dropna(subset=['aridity_proxy']).sort_values('year'))
        if len(sub) < 10:
            continue
        x = sub['year'].values.astype(float)
        y = sub['aridity_proxy'].values
        slope, _, r, p, _ = stats.linregress(x, y)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        period = f'{int(x.min())}-{int(x.max())}'
        print(f'{cl:6s} {ds_name:8s} {period:15s} {slope:+10.4f} {r:+6.3f} {p:8.4f} {sig}')
    print()

print('WA1 summer trend decomposition -- precip vs PET vs temp:')
print(f'{"Dataset":8s} {"Variable":25s} {"Slope/yr":10s} {"r":6s} {"p":8s} {"Sig"}')
print('-' * 65)
for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    sub = (df_lt[(df_lt['cluster'] == 'WA1') & (df_lt['dataset'] == ds_name)]
           .dropna(subset=['precip_sum','pet_sum']).sort_values('year'))
    if len(sub) < 10:
        continue
    for var, label in [('precip_sum', 'Summer precip (mm)'), ('pet_sum', 'Summer PET proxy (mm)'), ('tmean', 'Summer tmean (C)')]:
        x = sub['year'].values.astype(float)
        y = sub[var].dropna().values
        if len(y) < 10:
            continue
        slope, _, r, p, _ = stats.linregress(x[:len(y)], y)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{ds_name:8s} {label:25s} {slope:+10.4f} {r:+6.3f} {p:8.4f} {sig}')
    print()

df_lt.to_csv(f'{DROUGHT_DIR}/longterm_aridity_lista.csv', index=False)
print(f'Saved: {DROUGHT_DIR}/longterm_aridity_lista.csv')

## Noah's ideas: OGGM melt trend, mass-balance test, and replication check

### Does OGGM show increasing glacier melt over 2000-2019? (all clusters)

Generalized from an earlier WA1-only version: tests every cluster (not
just WA1) for a trend in annual glacier melt volume, to see whether the
long-term glacier-elevation warming found above (Section: long-term
aridity trend) translates into a detectable OGGM melt trend within the
shorter 2000-2019 study period, anywhere in Chile.

In [ ]:
print('Annual glacier melt trend 2000-2019 (OGGM), by cluster and dataset')
print('=' * 80)
print(f'{"Cluster":6s} {"Dataset":8s} {"Slope (m3/yr2)":15s} {"r":6s} {"p":8s} {"Sig"}')
print('-' * 80)

melt_trend_records = []

for cl in CLUSTERS:
    for ds_name in ['CR2MET', 'ERA5', 'CRU']:
        cl_gauges = [gid for gid, info in GAUGE_INFO_FRAC.items() if info['clusters'][0] == cl]

        ann_melt = np.zeros(len(YEARS))
        n_gauges = 0

        for gauge_id in cl_gauges:
            res = RESULTS[ds_name].get(gauge_id)
            if res is None:
                continue
            m  = res['monthly']
            cy = res.get('complete_years', YEARS)
            ann = (m[m['year'].isin(cy)].groupby('year')['melt_on_glacier_m3'].sum()
                   .reindex(YEARS, fill_value=np.nan))
            ann_melt += ann.fillna(0).values
            n_gauges += 1

        if n_gauges == 0:
            continue

        valid = ~np.isnan(ann_melt) & (ann_melt > 0)
        if valid.sum() < 8:
            continue
        x = np.array(YEARS)[valid].astype(float)
        y = ann_melt[valid]
        slope, intercept, r, p, _ = stats.linregress(x, y)
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{cl:6s} {ds_name:8s} {slope:+15.0f} {r:+6.3f} {p:8.4f} {sig}  (n={n_gauges} gauges)')

        melt_trend_records.append({'cluster': cl, 'dataset': ds_name, 'slope': slope,
                                   'r': r, 'p': p, 'sig': p < 0.05, 'n_gauges': n_gauges})

df_melt_trend_national = pd.DataFrame(melt_trend_records)
df_melt_trend_national.to_csv(f'{DROUGHT_DIR}/melt_trend_all_clusters.csv', index=False)

print()
print('Clusters with a significant melt trend in at least one dataset:')
sig_clusters = df_melt_trend_national[df_melt_trend_national['sig']]['cluster'].unique()
print(sig_clusters if len(sig_clusters) else 'None')

**Reflection:** whether or not any individual cluster shows a
significant OGGM melt trend over this shorter window, the broader point
holds: even without a clearly accelerating melt signal, glaciers can
still provide real hydrological stability during drought simply because
melt stays comparatively stable while precipitation-driven streamflow
collapses. The longer-term risk isn't necessarily a dramatic near-term
melt acceleration -- it's that glaciers eventually lose enough mass to
no longer sustain even current melt rates (the peak water problem). See
the mass-balance "bank account" test below for the more direct test of
this hypothesis.

### Mass balance trend vs. runoff trend -- the "bank account" test

SMB from `mb.csv` (OGGM calibrated, mm w.e./yr per glacier per year),
aggregated to basin level via area-weighted mean.

In [ ]:
from scipy.stats import linregress

print('Computing annual melt, precip-on-glacier, and Q per basin...')

trend_records = []

for gauge_id, info in GAUGE_INFO_FRAC.items():
    cy = info['complete_years']

    for ds_name in ['CR2MET', 'ERA5', 'CRU']:
        res = RESULTS[ds_name].get(gauge_id)
        if res is None:
            continue
        m = res['monthly']
        mc = m[m['year'].isin(cy)]

        ann = mc.groupby('year').agg(
            melt_m3=('melt_on_glacier_m3', 'sum'), liq_m3=('liq_prcp_on_glacier_m3', 'sum'),
            glac_m3=('glacier_runoff_m3', 'sum'), q_m3=('q_m3_mon', 'sum')).reset_index()
        ann = ann[ann['q_m3'] > 0]
        if len(ann) < 10:
            continue

        def trend(y_series, x_years):
            slope, intercept, r, p, se = linregress(x_years, y_series)
            mean_val = y_series.mean()
            pct_per_yr = (slope / mean_val * 100) if mean_val != 0 else np.nan
            cv = (y_series.std() / mean_val) if mean_val != 0 else np.nan
            return slope, p, pct_per_yr, cv

        yrs = ann['year'].values
        melt_slope, melt_p, melt_pct, melt_cv = trend(ann['melt_m3'].values, yrs)
        liq_slope,  liq_p,  liq_pct,  liq_cv  = trend(ann['liq_m3'].values,  yrs)
        glac_slope, glac_p, glac_pct, glac_cv = trend(ann['glac_m3'].values, yrs)
        q_slope,    q_p,    q_pct,    q_cv    = trend(ann['q_m3'].values,    yrs)

        trend_records.append({
            'gauge_id': gauge_id, 'cluster': info['clusters'][0], 'lat': info['cen_lat'],
            'dataset': ds_name, 'n_years': len(ann),
            'melt_pct_per_yr': melt_pct, 'melt_p': melt_p, 'melt_cv': melt_cv,
            'liq_pct_per_yr':  liq_pct,  'liq_p':  liq_p,  'liq_cv':  liq_cv,
            'glac_pct_per_yr': glac_pct, 'glac_p': glac_p, 'glac_cv': glac_cv,
            'q_pct_per_yr':    q_pct,    'q_p':    q_p,    'q_cv':    q_cv,
        })

df_trends = pd.DataFrame(trend_records)
df_trends.to_csv(f'{DROUGHT_DIR}/melt_precip_trends_by_basin.csv', index=False)
print(f'Trend records: {len(df_trends):,} ({df_trends["gauge_id"].nunique()} basins)')

print()
print('Median annual trend (%/yr) by cluster -- CR2MET (sig% = fraction basins p<0.05)')
print('='*100)
print(f'{"Cluster":6s} {"n":4s} {"Melt %/yr":10s} {"sig%":6s}  {"Precip %/yr":11s} {"sig%":6s}  '
      f'{"Runoff %/yr":11s} {"sig%":6s}  {"Q %/yr":8s} {"sig%":6s}')
print('-'*100)

cr2 = df_trends[df_trends['dataset'] == 'CR2MET']
for cl in CLUSTERS:
    sub = cr2[cr2['cluster'] == cl]
    if len(sub) == 0:
        continue
    n = len(sub)

    def summarize(pct_col, p_col):
        return sub[pct_col].median(), (sub[p_col] < 0.05).mean() * 100

    melt_med, melt_sig = summarize('melt_pct_per_yr', 'melt_p')
    liq_med,  liq_sig  = summarize('liq_pct_per_yr',  'liq_p')
    glac_med, glac_sig = summarize('glac_pct_per_yr', 'glac_p')
    q_med,    q_sig    = summarize('q_pct_per_yr',    'q_p')

    print(f'{cl:6s} {n:4d} {melt_med:+10.2f} {melt_sig:5.0f}%  {liq_med:+11.2f} {liq_sig:5.0f}%  '
          f'{glac_med:+11.2f} {glac_sig:5.0f}%  {q_med:+8.2f} {q_sig:5.0f}%')

print()
print('INTERPRETATION: Melt %/yr > 0 significant = ice melt volume increasing. Runoff')
print('%/yr near 0 despite melt trending up = "draining the bank account" pattern.')

### Supporting evidence: melt/precip-on-glacier trend table (Noah)

The per-cluster trend table below is the evidence base for the
mass-balance "bank account" classification just above -- it shows,
per cluster, the median annual trend (%/yr) and significance share for
ice melt, precip-on-glacier, total glacier runoff, and streamflow.

In [ ]:
from scipy.stats import linregress
rgi_areas = pd.read_csv(RGI_PATH)[['RGIId', 'Area']].set_index('RGIId')

print('Loading per-glacier SMB (mb.csv) across all clusters/datasets...')

smb_by_glacier = {}

for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    smb_by_glacier[ds_name] = {}
    for cl in CLUSTERS:
        mb_file = f'{OUTPUT_ROOT}/{ds_name}/{cl}/mb.csv'
        if not os.path.exists(mb_file):
            continue
        mb_df = pd.read_csv(mb_file).set_index('rgi_id')
        year_cols = mb_df.columns[:20]
        for rgi_id, row in mb_df.iterrows():
            vals = row[year_cols].values.astype(float)
            if np.all(np.isnan(vals)):
                continue
            smb_by_glacier[ds_name][rgi_id] = pd.Series(vals, index=YEARS)

for ds_name in smb_by_glacier:
    print(f'  {ds_name}: {len(smb_by_glacier[ds_name])} glaciers with SMB')

print('Computing basin-level annual SMB (area-weighted)...')

smb_trend_records = []

for gauge_id, info in GAUGE_INFO_FRAC.items():
    rgi_ids = info.get('rgi_ids', [])
    if not rgi_ids:
        continue

    for ds_name in ['CR2MET', 'ERA5', 'CRU']:
        avail = [r for r in rgi_ids if r in smb_by_glacier[ds_name]]
        avail = [r for r in avail if r in rgi_areas.index]
        if len(avail) == 0:
            continue

        weights = rgi_areas.loc[avail, 'Area'].values
        smb_matrix = np.array([smb_by_glacier[ds_name][r].values for r in avail])

        ann_smb = np.average(smb_matrix, axis=0, weights=weights)
        ann_smb = pd.Series(ann_smb, index=YEARS).dropna()

        if len(ann_smb) < 10:
            continue

        slope, intercept, r, p, se = linregress(ann_smb.index.values, ann_smb.values)

        smb_trend_records.append({
            'gauge_id': gauge_id, 'cluster': info['clusters'][0], 'lat': info['cen_lat'],
            'dataset': ds_name, 'n_glaciers': len(avail), 'n_years': len(ann_smb),
            'smb_mean': ann_smb.mean(), 'smb_slope': slope, 'smb_p': p,
        })

df_smb_trends = pd.DataFrame(smb_trend_records)
df_smb_trends.to_csv(f'{DROUGHT_DIR}/smb_trends_by_basin.csv', index=False)
print(f'SMB trend records: {len(df_smb_trends):,} ({df_smb_trends["gauge_id"].nunique()} basins)')

merged_trends = df_trends.merge(
    df_smb_trends[['gauge_id','dataset','smb_mean','smb_slope','smb_p']], on=['gauge_id','dataset'], how='inner')

print()
print('SMB trend vs. glacier runoff trend by cluster -- CR2MET')
print('='*95)
print(f'{"Cluster":6s} {"n":4s} {"SMB mean":10s} {"SMB slope":11s} {"sig%":6s}  {"Runoff %/yr":11s} {"sig%":6s}  {"Pattern"}')
print('-'*95)

cr2_merged = merged_trends[merged_trends['dataset'] == 'CR2MET']
for cl in CLUSTERS:
    sub = cr2_merged[cr2_merged['cluster'] == cl]
    if len(sub) == 0:
        continue
    n = len(sub)
    smb_mean_med  = sub['smb_mean'].median()
    smb_slope_med = sub['smb_slope'].median()
    smb_sig       = (sub['smb_p'] < 0.05).mean() * 100
    glac_med      = sub['glac_pct_per_yr'].median()
    glac_sig      = (sub['glac_p'] < 0.05).mean() * 100

    if smb_slope_med < 0 and smb_sig > 30 and abs(glac_med) < 1.5:
        pattern = 'BANK ACCOUNT (MB down, runoff flat)'
    elif smb_slope_med < 0 and glac_med > 0:
        pattern = 'MB down, runoff UP (partial buffering)'
    elif smb_slope_med < 0 and glac_med < 0:
        pattern = 'MB down, runoff down (declining together)'
    else:
        pattern = '-'

    print(f'{cl:6s} {n:4d} {smb_mean_med:10.1f} {smb_slope_med:11.2f} {smb_sig:5.0f}%  '
          f'{glac_med:+11.2f} {glac_sig:5.0f}%  {pattern}')

## Addressing Noah's ideas, continued: replication check and mass-balance decomposition

(The melt/precip trend table and mass-balance "bank account" test are
above. This continues with a literature replication check and a finer
accumulation-vs-ablation decomposition of the mass-balance trend.)

### Direct replication check -- Ayala et al. framing (Maipo)

Ayala et al. compared period MEANS (2000-2009 vs 2010-2019), not a
continuous linear trend. Reproduces that exact comparison for Rio Maipo
En El Manzano to check consistency with the published "runoff almost
unaltered, sustained by ~10% ice volume loss" finding, plus all of DA2
for context.

In [ ]:
MAIPO_GAUGE = '5710001'

def period_mean_check(gauge_id, dataset='CR2MET'):
    info = GAUGE_INFO_FRAC.get(gauge_id)
    if info is None:
        return None
    cy = info['complete_years']
    res = RESULTS[dataset].get(gauge_id)
    if res is None:
        return None
    m = res['monthly']
    mc = m[m['year'].isin(cy)]

    ann = mc.groupby('year').agg(melt_m3=('melt_on_glacier_m3', 'sum'), liq_m3=('liq_prcp_on_glacier_m3', 'sum'),
                                 glac_m3=('glacier_runoff_m3', 'sum'), q_m3=('q_m3_mon', 'sum')).reset_index()
    ann = ann[ann['q_m3'] > 0]

    base = ann[ann['year'].isin(range(2000,2010))]
    drgt = ann[ann['year'].isin(range(2010,2020))]
    if len(base) == 0 or len(drgt) == 0:
        return None

    def pct_chg(col):
        b, d = base[col].mean(), drgt[col].mean()
        return b, d, (d - b) / b * 100 if b != 0 else np.nan

    return {'gauge_id': gauge_id, 'melt': pct_chg('melt_m3'), 'liq': pct_chg('liq_m3'),
            'runoff': pct_chg('glac_m3'), 'q': pct_chg('q_m3')}

print('Period-mean comparison (Ayala-style): 2000-2009 vs 2010-2019')
print('='*70)

result = period_mean_check(MAIPO_GAUGE)
if result:
    print(f'Rio Maipo En El Manzano ({MAIPO_GAUGE}) -- CR2MET')
    for key, label in [('melt','Ice melt'), ('liq','Precip on glacier'), ('runoff','Total glacier runoff'), ('q','Q')]:
        b, d, pct = result[key]
        print(f'  {label:22s}  base={b/1e6:8.2f}M  drought={d/1e6:8.2f}M  delta={pct:+.1f}%')
else:
    print(f'No data for {MAIPO_GAUGE} (not in the final basin set)')

print()
print('All DA2 basins, individually, for context:')
print(f'{"gauge_id":12s} {"melt d%":9s} {"liq d%":8s} {"runoff d%":10s} {"Q d%"}')
print('-'*55)
for gid, info in GAUGE_INFO_FRAC.items():
    if info['clusters'][0] != 'DA2':
        continue
    r = period_mean_check(gid)
    if r is None:
        continue
    print(f'{gid:12s} {r["melt"][2]:+8.1f}% {r["liq"][2]:+7.1f}% {r["runoff"][2]:+9.1f}% {r["q"][2]:+.1f}%')

### Decompose MB trend into accumulation vs. ablation drivers

MB = accumulation (snowfall_on_glacier) - ablation (melt_on_glacier).
Decomposes the CHANGE in MB (2010-2019 vs 2000-2009) into how much is
explained by declining accumulation vs. increasing ablation, using the
same counterfactual approach as the Q-decline-vs-melt-change
decomposition above.

In [ ]:
import xarray as xr
print('Extracting annual snowfall_on_glacier and melt_on_glacier per basin (CR2MET)...')

accum_ablat_records = []

for ds_name in ['CR2MET']:
    for cl in CLUSTERS:
        nc_file = f'{OUTPUT_ROOT}/{ds_name}/{cl}/run_output_2000_2019_hydro_TC_{cl}.nc'
        if not os.path.exists(nc_file):
            continue

        ds = xr.open_dataset(nc_file)
        rgi_all = list(ds.rgi_id.values)
        actual_years = ds['time'].values.astype(int).tolist()

        snow_annual = np.nansum(ds['snowfall_on_glacier_monthly'].values, axis=1) / WATER_DENSITY
        melt_annual = np.nansum(ds['melt_on_glacier_monthly'].values, axis=1) / WATER_DENSITY

        if snow_annual.shape[0] != len(actual_years):
            ds.close()
            continue

        for gauge_id, info in GAUGE_INFO_FRAC.items():
            if info['clusters'][0] != cl:
                continue
            rgi_ids = info.get('rgi_ids', [])
            mask_ids = [r for r in rgi_ids if r in rgi_all]
            if not mask_ids:
                continue
            mask = np.isin(rgi_all, mask_ids)

            basin_snow = snow_annual[:, mask].sum(axis=1)
            basin_melt = melt_annual[:, mask].sum(axis=1)

            accum_ablat_records.append({'gauge_id': gauge_id, 'cluster': cl,
                                        'years': actual_years, 'snow_m3': basin_snow, 'melt_m3': basin_melt})

        ds.close()

print(f'Extracted for {len(accum_ablat_records)} basin records')

decomp_mb_records = []

for rec in accum_ablat_records:
    ann = pd.DataFrame({'year': rec['years'], 'snow_m3': rec['snow_m3'], 'melt_m3': rec['melt_m3']})
    base = ann[ann['year'].isin(range(2000, 2010))]
    drgt = ann[ann['year'].isin(range(2010, 2020))]
    if len(base) == 0 or len(drgt) == 0:
        continue

    snow_base, melt_base = base['snow_m3'].mean(), base['melt_m3'].mean()
    mb_base = snow_base - melt_base
    snow_dr, melt_dr = drgt['snow_m3'].mean(), drgt['melt_m3'].mean()
    mb_dr = snow_dr - melt_dr

    delta_mb = mb_dr - mb_base
    if abs(delta_mb) < 1e-6:
        continue

    mb_cf_snow_only = snow_dr   - melt_base
    mb_cf_melt_only = snow_base - melt_dr

    pct_accum_driven = (mb_cf_snow_only - mb_base) / delta_mb * 100
    pct_ablat_driven = (mb_cf_melt_only - mb_base) / delta_mb * 100

    decomp_mb_records.append({
        'gauge_id': rec['gauge_id'], 'cluster': rec['cluster'],
        'snow_chg_pct': (snow_dr - snow_base) / snow_base * 100 if snow_base > 0 else np.nan,
        'melt_chg_pct': (melt_dr - melt_base) / melt_base * 100 if melt_base > 0 else np.nan,
        'delta_mb_m3': delta_mb,
        'pct_accum_driven': np.clip(pct_accum_driven, -200, 200),
        'pct_ablat_driven': np.clip(pct_ablat_driven, -200, 200),
    })

df_mb_decomp = pd.DataFrame(decomp_mb_records)
df_mb_decomp.to_csv(f'{DROUGHT_DIR}/mb_decomposition_by_basin.csv', index=False)

print()
print('MB change decomposition by cluster (2010-2019 vs 2000-2009), only worsening-MB basins')
print('='*80)
print(f'{"Cluster":6s} {"n":4s} {"Snow d%":9s} {"Melt d%":9s}  {"% due to less accum":19s} {"% due to more ablat"}')
print('-'*80)

for cl in CLUSTERS:
    sub = df_mb_decomp[(df_mb_decomp['cluster'] == cl) & (df_mb_decomp['delta_mb_m3'] < 0)]
    if len(sub) == 0:
        continue
    n = len(sub)
    print(f'{cl:6s} {n:4d} {sub["snow_chg_pct"].median():+8.1f}% {sub["melt_chg_pct"].median():+8.1f}%  '
          f'{sub["pct_accum_driven"].median():18.1f}%  {sub["pct_ablat_driven"].median():18.1f}%')

## Sanity checks (kept for reference, not part of the main analysis flow)

The two cells below (mu_star comparison, seasonal-definition test) are
useful methodological sanity checks but don't feed into the drought/
recession/aridity/lag narrative above -- kept at the end for reference.

## Ale's question: mu_star (degree-day factor) comparison across datasets and clusters

In [ ]:
import json as jsonlib

def load_mustar_cluster(ds_name, cluster):
    base = f'{OUTPUT_ROOT}/{ds_name}/{cluster}/per_glacier'
    if not os.path.exists(base):
        return np.nan, 0
    mu_vals = []
    for root, dirs, files in os.walk(base):
        for fname in files:
            if fname == 'local_mustar.json':
                try:
                    with open(os.path.join(root, fname)) as f:
                        d = jsonlib.load(f)
                    val = d.get('mu_star_glacierwide', None)
                    if val is not None and not np.isnan(float(val)):
                        mu_vals.append(float(val))
                except Exception:
                    pass
    if not mu_vals:
        return np.nan, 0
    return np.median(mu_vals), len(mu_vals)

print('Loading mu_star_glacierwide for all clusters and datasets...')
print('(This will take a few minutes -- reading one JSON per glacier)')
print()

mustar_results = {}
for ds_name in ['CR2MET', 'ERA5', 'CRU']:
    mustar_results[ds_name] = {}
    for cl in CLUSTERS:
        med, n = load_mustar_cluster(ds_name, cl)
        mustar_results[ds_name][cl] = med
        print(f'  {ds_name} {cl}: median mu* = {med:.2f} mm C-1 month-1 (n={n} glaciers)')
    print()

print('Median mu* (mm C-1 month-1) by cluster and dataset:')
print(f'{"Cluster":6s}  {"CR2MET":10s}  {"ERA5":10s}  {"CRU":10s}  {"ERA5/CR2":10s}  {"CRU/CR2"}')
print('-' * 65)
for cl in CLUSTERS:
    cr2 = mustar_results['CR2MET'].get(cl, np.nan)
    era = mustar_results['ERA5'].get(cl, np.nan)
    cru = mustar_results['CRU'].get(cl, np.nan)
    r_era = era/cr2 if cr2 > 0 else np.nan
    r_cru = cru/cr2 if cr2 > 0 else np.nan
    print(f'{cl:6s}  {cr2:10.2f}  {era:10.2f}  {cru:10.2f}  {r_era:10.2f}  {r_cru:.2f}')

print()
print('mu* in daily units (mm d-1 C-1, divide monthly by 30.4):')
print('Literature range: 3-10 mm d-1 C-1 (Hock, 2003)')
print(f'{"Cluster":6s}  {"CR2MET":10s}  {"ERA5":10s}  {"CRU":10s}')
print('-' * 40)
for cl in CLUSTERS:
    cr2 = mustar_results['CR2MET'].get(cl, np.nan) / 30.4
    era = mustar_results['ERA5'].get(cl, np.nan)   / 30.4
    cru = mustar_results['CRU'].get(cl, np.nan)    / 30.4
    print(f'{cl:6s}  {cr2:10.3f}  {era:10.3f}  {cru:10.3f}')

## What is "summer"? Testing seasonal definitions

Compares Annual, Ablation (Oct-Mar), Summer (Feb-Apr), February, and
March glacier fractions. Finding: peak ablation falls somewhere between
March and April, not DJF/JFM -- suggests using **MMA (Mar-Apr-May)** or
similar rather than the conventional DJF definition of summer.

In [ ]:
OCT_MAR = [10, 11, 12, 1, 2, 3]
FEB_APR = [2, 3, 4]
MARCH   = [3]

print('Seasonal glacier fraction summary -- all basins (CR2MET)')
print('=' * 65)

for season_name, months in [
    ('Annual (all months)', list(range(1,13))),
    ('Ablation (Oct-Mar)',  OCT_MAR),
    ('Summer (Feb-Apr)',    FEB_APR),
    ('February',            [2]),
    ('March',               MARCH),
]:
    fracs = []
    for gauge_id, res in RESULTS['CR2MET'].items():
        if res is None:
            continue
        info = GAUGE_INFO_FRAC.get(gauge_id)
        if info is None:
            continue
        m  = res['monthly']
        cy = res.get('complete_years', YEARS)
        sub = m[m['year'].isin(cy) & m['month'].isin(months)]
        sub = sub[sub['q_m3_mon'] > 0]
        if len(sub) == 0:
            continue
        ann = sub.groupby('year').agg(glac=('glacier_runoff_m3', 'sum'), q=('q_m3_mon', 'sum'))
        ann = ann[ann['q'] > 0]
        frac = (ann['glac'] / ann['q']).mean() * 100
        if not np.isnan(frac) and frac <= 100:
            fracs.append(frac)

    fracs = np.array(fracs)
    print(f'{season_name} (n={len(fracs)}): min={fracs.min():.2f}%  median={np.median(fracs):.2f}%  '
          f'mean={fracs.mean():.2f}%  max={fracs.max():.2f}%')

print()
print('Cluster means by season (CR2MET):')
print(f'{"Cluster":6s} {"Annual":8s} {"Oct-Mar":8s} {"Feb-Apr":8s} {"March":8s}')
print('-' * 40)

for cl in CLUSTERS:
    cl_gauges = [gid for gid, info in GAUGE_INFO_FRAC.items() if info['clusters'][0] == cl]
    row = [cl]
    for months in [list(range(1,13)), OCT_MAR, FEB_APR, MARCH]:
        fracs = []
        for gid in cl_gauges:
            res = RESULTS['CR2MET'].get(gid)
            if res is None:
                continue
            m  = res['monthly']
            cy = res.get('complete_years', YEARS)
            sub = m[m['year'].isin(cy) & m['month'].isin(months)]
            sub = sub[sub['q_m3_mon'] > 0]
            if len(sub) == 0:
                continue
            ann = sub.groupby('year').agg(glac=('glacier_runoff_m3','sum'), q=('q_m3_mon','sum'))
            ann = ann[ann['q'] > 0]
            frac = (ann['glac']/ann['q']).mean()*100
            if not np.isnan(frac) and frac <= 100:
                fracs.append(frac)
        row.append(f'{np.mean(fracs):.1f}%' if fracs else 'n/a')
    print(f'{row[0]:6s} {row[1]:8s} {row[2]:8s} {row[3]:8s} {row[4]:8s}')

print()
print('Result: across all three datasets, roughly half of basins show statistically')
print('significant INCREASING glacier fraction trends over 2000-2019 (only ~1 basin')
print('per dataset shows significant decrease). WA1: ~68% significant, median slope')
print('~+0.07 pp/yr, concentrated in Aconcagua/Maipo/Bio-Bio -- Santiago and central')
print('Chile ag heartland supply basins.')
print()
print('CAVEAT: the 2000-2019 trend period is almost entirely coincident with the')
print('2010-2019 megadrought. The increasing trend may largely reflect the megadrought')
print('rather than an independent secular trend -- resolving this needs data beyond')
print('2019 or a longer pre-drought baseline. Include this caveat in the paper.')

## Summary of findings

**Everything below is from the OLD run (143 nested basins). Rewrite from the new output before citing.**

### Old summary text

**1. Empirical validation of drought period definitions**

The megadrought was not a Chile-wide phenomenon. Precipitation deficits
concentrated in central/northern Chile (DA2: -37 to -40%, DA3: -28 to
-30%, WA1: -16 to -17%), while Patagonia (WA3, WA4, WA6) received
above-normal precipitation over the same periods -- consistent with
Garreaud et al. (2020)'s characterization of this as a central-Chile
event.

**2. Snow drought classification**

Two distinct mechanisms: dry snow drought (DA1, DA2, DA3, WA1) and warm
snow drought (WA2, WA3, WA4, WA6). DA3 and WA1 transitioned from dry to
compound (dry+warm) by 2015-2019.

**3. Glacier fraction amplification and decomposition -- basin-set and
dataset dependence**

Sections IV, V, and VII (comparison figure) show these numbers directly
depend on both basin-set choice and climate dataset -- see the printed
tables and the basin-set comparison figure above for the actual
sensitivity, rather than a single headline number. Use those tables when
deciding what to report to co-authors and in the paper, rather than
citing one basin-set/dataset combination in isolation.

**4. Climate drivers**

Temperature anomalies during drought were modest (+0.07 to +0.42C)
compared to precipitation deficits (-55% in DA2). Precipitation decline
is the dominant driver of drought amplification in central Chile.

### Pending analyses (continued in 06_megadrought_extensions.ipynb)

- Peak water analysis, recession/routing (Becca), basin aridity (Becca),
  mu_star comparison (Ale), seasonal-definition question, corrected WA1
  lag analysis, long-term melt/mass-balance trends and the "bank
  account" hypothesis (Noah), and methods-draft statistics (Ale).